<a href="https://colab.research.google.com/github/RamaraoD423/GenAI_L3---Bronze/blob/main/RAGSystemApplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Configuration
DATA_DIR = "./data"
DB_DIR = "./vector_store"
MODEL_NAME = "llama3.2"
EMBEDDING_MODEL = "nomic-embed-text"

def initialize_vector_store():
    """Loads all PDFs from directory, chunks them, and saves to Chroma DB."""
    print("Loading PDF documents from directory...")
    loader = PyPDFDirectoryLoader(DATA_DIR)
    docs = loader.load()

    if not docs:
        raise ValueError(f"No PDF files found in {DATA_DIR}. Please add at least 3 PDFs.")

    print(f"Loaded {len(docs)} total pages. Splitting into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = text_splitter.split_documents(docs)

    print("Generating local embeddings and creating vector store...")
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_DIR
    )
    print("Vector database built and saved locally!")
    return vector_store

def get_rag_chain(vector_store):
    """Builds the LCEL RAG chain using Llama 3.2."""
    retriever = vector_store.as_retriever(search_kwargs={"k": 4})
    llm = ChatOllama(model=MODEL_NAME)

    # Prompt template enforcing strict grounding to context
    template = """Answer the question strictly using only the provided context below.
If the answer cannot be found in the context, say "I cannot find the answer in the provided documents."

Context:
{context}

Question: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

def main():
    # Ensure data folder exists
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)
        print(f"Created '{DATA_DIR}' folder. Please place your 3+ PDF documents inside it and rerun.")
        return

    # Load or create vector database
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    if os.path.exists(DB_DIR) and os.listdir(DB_DIR):
        print("Loading existing vector store from disk...")
        vector_store = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    else:
        vector_store = initialize_vector_store()

    rag_chain = get_rag_chain(vector_store)

    print("\n--- RAG Application Running (Llama 3.2 via Ollama) ---")
    print("Ask questions about your PDFs. Type 'exit' to quit.\n")

    while True:
        query = input("Query: ")
        if query.lower() == 'exit':
            break
        if not query.strip():
            continue

        print("\nThinking...")
        response = rag_chain.invoke(query)
        print(f"\nAnswer:\n{response}\n" + "-"*50)

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Configuration
DATA_DIR = "./data"
DB_DIR = "./vector_store"
MODEL_NAME = "llama3.2"
EMBEDDING_MODEL = "nomic-embed-text"

def initialize_vector_store():
    """Loads all PDFs from directory, chunks them, and saves to Chroma DB."""
    print("Loading PDF documents from directory...")
    loader = PyPDFDirectoryLoader(DATA_DIR)
    docs = loader.load()

    if not docs:
        raise ValueError(f"No PDF files found in {DATA_DIR}. Please add at least 3 PDFs.")

    print(f"Loaded {len(docs)} total pages. Splitting into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = text_splitter.split_documents(docs)

    print("Generating local embeddings and creating vector store...")
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_DIR
    )
    print("Vector database built and saved locally!")
    return vector_store

def get_rag_chain(vector_store):
    """Builds the LCEL RAG chain using Llama 3.2."""
    retriever = vector_store.as_retriever(search_kwargs={"k": 4})
    llm = ChatOllama(model=MODEL_NAME)

    # Prompt template enforcing strict grounding to context
    template = """Answer the question strictly using only the provided context below.
If the answer cannot be found in the context, say "I cannot find the answer in the provided documents."

Context:
{context}

Question: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

def main():
    # Ensure data folder exists
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)
        print(f"Created '{DATA_DIR}' folder. Please place your 3+ PDF documents inside it and rerun.")
        return

    # Load or create vector database
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    if os.path.exists(DB_DIR) and os.listdir(DB_DIR):
        print("Loading existing vector store from disk...")
        vector_store = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    else:
        vector_store = initialize_vector_store()

    rag_chain = get_rag_chain(vector_store)

    print("\n--- RAG Application Running (Llama 3.2 via Ollama) ---")
    print("Ask questions about your PDFs. Type 'exit' to quit.\n")

    while True:
        query = input("Query: ")
        if query.lower() == 'exit':
            break
        if not query.strip():
            continue

        print("\nThinking...")
        response = rag_chain.invoke(query)
        print(f"\nAnswer:\n{response}\n" + "-"*50)

if __name__ == "__main__":
    main()